In [1]:
# ============================================================
# 06_unified_ecm.ipynb | Cell 1
# Composite Equilibrium + ECT Construction + ADF Gate
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from statsmodels.tsa.stattools import adfuller

PROCESSED = '../data/processed/'

# ── Load all phase outputs + lagged master ──────────────────────
ml = pd.read_csv(PROCESSED+'master_v2_lagged.csv', index_col=0, parse_dates=True)
ml.index = pd.to_datetime(ml.index).to_period('M').to_timestamp()

p2 = pd.read_csv(PROCESSED+'phase2_equilibrium_v2.csv', index_col=0, parse_dates=True)
p2.index = pd.to_datetime(p2.index).to_period('M').to_timestamp()

p3 = pd.read_csv(PROCESSED+'phase3_feer_v2.csv', index_col=0, parse_dates=True)
p3.index = pd.to_datetime(p3.index).to_period('M').to_timestamp()

p4 = pd.read_csv(PROCESSED+'phase4_beer_v2.csv', index_col=0, parse_dates=True)
p4.index = pd.to_datetime(p4.index).to_period('M').to_timestamp()

# ── 1) REER fair (monthly, lag=0 → use directly) ───────────────
reer_fair = p2['inr_fair_reer'].dropna()

# ── 2) FEER static fair (quarterly → shift 6mo pub lag → fwd-fill)
feer_raw = p3['inr_fair_static'].dropna()
feer_shifted = feer_raw.copy()
feer_shifted.index = feer_shifted.index + pd.DateOffset(months=6)
grid = pd.date_range(feer_shifted.index.min(), ml.index.max(), freq='MS')
feer_monthly = feer_shifted.reindex(grid).ffill()

# ── 3) Composite: 50/50 where both available; REER-only elsewhere
feer_avail = feer_monthly.dropna()
composite = pd.Series(dtype=float, index=reer_fair.index)
for dt in reer_fair.index:
    r = reer_fair[dt]
    composite[dt] = 0.5*r + 0.5*feer_avail[dt] if dt in feer_avail.index else r
composite = composite.dropna()

# ── 4) ECT = log(INR_actual) - log(composite_fair) ─────────────
# positive ECT = INR weaker than fair (undervalued)
# negative ECT = INR stronger than fair (overvalued)
inr = ml['inr_usd'].reindex(composite.index).dropna()
common = inr.index.intersection(composite.index)

ecm = pd.DataFrame({
    'inr_usd': inr.loc[common],
    'composite_fair': composite.loc[common],
    'reer_fair': reer_fair.reindex(common),
    'feer_fair': feer_monthly.reindex(common),
}, index=common).dropna()
ecm['log_inr']  = np.log(ecm['inr_usd'])
ecm['log_fair'] = np.log(ecm['composite_fair'])
ecm['ect']      = ecm['log_inr'] - ecm['log_fair']

# Add regime probability from Phase 4
ecm['p_stress'] = p4['p_stress'].reindex(ecm.index)

# Pre-build short-run drivers (Δ terms) for Cell 2
ecm['d_log_inr']   = ecm['log_inr'].diff()
ecm['d_log_dxy']   = np.log(ml['dxy']).reindex(ecm.index).diff()
ecm['d_log_brent'] = np.log(ml['brent']).reindex(ecm.index).diff()
ecm['d_rrd']       = ml['real_rate_diff'].reindex(ecm.index).diff()
ecm['d_vix']       = ml['vix'].reindex(ecm.index).diff()
ecm['d_fpi']       = ml['fpi_pct_gdp'].reindex(ecm.index).diff()
ecm['ect_lag1']    = ecm['ect'].shift(1)

print("="*65)
print("CELL 1 — COMPOSITE EQUILIBRIUM + ECT")
print("="*65)
print(f"REER fair     : {reer_fair.index[0].strftime('%b%Y')} → {reer_fair.index[-1].strftime('%b%Y')}  (monthly)")
print(f"FEER fair     : shifted +6mo, fwd-filled to monthly")
print(f"Composite     : 50/50 REER+FEER where both available")
print(f"ECM dataset   : {ecm.index[0].strftime('%b%Y')} → {ecm.index[-1].strftime('%b%Y')}  (n={len(ecm)})")

# ── 5) ADF gate ────────────────────────────────────────────────
adf = adfuller(ecm['ect'].dropna(), maxlag=12, autolag='AIC')
print(f"\n{'='*65}")
print(f"ADF GATE — is ECT stationary?")
print(f"{'='*65}")
print(f"  ADF stat  = {adf[0]:.3f}")
print(f"  p-value   = {adf[1]:.4f}")
print(f"  Lags      = {adf[2]}")
print(f"  Critical  : 1%={adf[4]['1%']:.3f}  5%={adf[4]['5%']:.3f}")
gate = adf[1] < 0.05
print(f"  {'✓ GO — ECT is stationary. Proceed to ECM.' if gate else '✗ STOP — ECT NOT stationary. Cannot build ECM.'}")

# ── 6) ECT diagnostics ─────────────────────────────────────────
e = ecm['ect'].dropna()
print(f"\nECT DIAGNOSTICS:")
print(f"  Mean:     {e.mean():+.4f}  (non-zero ≈ RBI managed-float premium)")
print(f"  Std:      {e.std():.4f}")
print(f"  Range:    {e.min():+.4f} ({e.idxmin().strftime('%b%Y')}) to"
      f" {e.max():+.4f} ({e.idxmax().strftime('%b%Y')})")
print(f"  Current:  {e.iloc[-1]:+.4f}  → INR is {abs(e.iloc[-1])*100:.1f}%"
      f" {'weaker' if e.iloc[-1]>0 else 'stronger'} than composite fair")

print(f"\nECT AT KEY EPISODES:")
for lbl, dt_str in [('GFC Oct08','2008-10-01'), ('Taper Aug13','2013-08-01'),
                      ('COVID Apr20','2020-04-01'), ('Fed tight Oct22','2022-10-01'),
                      ('Pre-war Dec25','2025-12-01'), ('Latest','')]:
    dt = pd.Timestamp(dt_str) if dt_str else ecm.index[-1]
    if dt in ecm.index:
        r = ecm.loc[dt]
        print(f"  {lbl:<16} INR={r['inr_usd']:.2f}  fair={r['composite_fair']:.2f}"
              f"  ECT={r['ect']:+.4f} ({abs(r['ect'])*100:.1f}% {'weak' if r['ect']>0 else 'strong'})")

ecm.to_csv(PROCESSED+'ecm_base.csv')
print(f"\n✓ Saved: ecm_base.csv ({ecm.shape[0]} × {ecm.shape[1]})")
print(f"  Columns: {list(ecm.columns)}")
print(f"\n  Ready for Cell 2: ECM estimation")

CELL 1 — COMPOSITE EQUILIBRIUM + ECT
REER fair     : Apr2004 → Apr2026  (monthly)
FEER fair     : shifted +6mo, fwd-filled to monthly
Composite     : 50/50 REER+FEER where both available
ECM dataset   : Apr2004 → Apr2026  (n=265)

ADF GATE — is ECT stationary?
  ADF stat  = -5.058
  p-value   = 0.0000
  Lags      = 12
  Critical  : 1%=-3.457  5%=-2.873
  ✓ GO — ECT is stationary. Proceed to ECM.

ECT DIAGNOSTICS:
  Mean:     +0.0235  (non-zero ≈ RBI managed-float premium)
  Std:      0.0435
  Range:    -0.0736 (Jan2011) to +0.1537 (Oct2008)
  Current:  +0.0825  → INR is 8.2% weaker than composite fair

ECT AT KEY EPISODES:
  GFC Oct08        INR=48.64  fair=41.71  ECT=+0.1537 (15.4% weak)
  Taper Aug13      INR=63.21  fair=56.47  ECT=+0.1127 (11.3% weak)
  COVID Apr20      INR=76.24  fair=70.93  ECT=+0.0722 (7.2% weak)
  Fed tight Oct22  INR=82.34  fair=77.88  ECT=+0.0557 (5.6% weak)
  Pre-war Dec25    INR=90.09  fair=82.73  ECT=+0.0852 (8.5% weak)
  Latest           INR=93.55  fair=86

In [2]:
# ============================================================
# 06_unified_ecm.ipynb | Cell 2
# Tier 2+3: ECM Estimation + Regime Conditioning
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_breusch_godfrey

PROCESSED = '../data/processed/'
ecm = pd.read_csv(PROCESSED+'ecm_base.csv', index_col=0, parse_dates=True)
ecm.index = pd.to_datetime(ecm.index).to_period('M').to_timestamp()

# ── TIER 2: Basic 1-month ECM ──────────────────────────────────
dep = 'd_log_inr'
regressors = ['ect_lag1','d_log_dxy','d_log_brent','d_rrd','d_vix']
d = ecm[[dep]+regressors].dropna()

print("="*65)
print("TIER 2 — BASIC 1-MONTH ECM")
print("="*65)
print(f"Sample: {d.index[0].strftime('%b%Y')} → {d.index[-1].strftime('%b%Y')}  (n={len(d)})")

X = sm.add_constant(d[regressors]); y = d[dep]
res1 = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':6})
print(f"  R² = {res1.rsquared:.3f}")
print(f"  {'variable':<14}{'coef':>10}{'t-stat':>8}{'p-val':>8}")
for v in ['const']+regressors:
    sig = '***' if res1.pvalues[v]<0.01 else ('**' if res1.pvalues[v]<0.05 else ('*' if res1.pvalues[v]<0.1 else ''))
    print(f"  {v:<14}{res1.params[v]:>+10.5f}{res1.tvalues[v]:>8.2f}{res1.pvalues[v]:>8.4f} {sig}")

alpha = res1.params['ect_lag1']
print(f"\n  α = {alpha:.4f} (p={res1.pvalues['ect_lag1']:.3f})")
print(f"  ⚠ Insignificant at monthly frequency — consistent with Meese-Rogoff.")
print(f"  Testing longer horizons and regime conditioning...\n")

# ── MULTI-HORIZON PREDICTABILITY ────────────────────────────────
print("="*65)
print("MULTI-HORIZON: Δlog(INR)_{{t:t+h}} = c + α_h · ECT_t")
print("="*65)
print(f"  {'h (months)':>10}{'α':>10}{'t-stat':>8}{'p-val':>8}{'R²':>8}")
horizons = {}
for h in [1, 3, 6, 9, 12]:
    fwd = ecm['log_inr'].shift(-h) - ecm['log_inr']
    dd = pd.DataFrame({'fwd': fwd, 'ect': ecm['ect']}).dropna()
    X = sm.add_constant(dd['ect']); yy = dd['fwd']
    r = sm.OLS(yy, X).fit(cov_type='HAC', cov_kwds={'maxlags':max(h,6)})
    sig = '***' if r.pvalues['ect']<0.01 else ('**' if r.pvalues['ect']<0.05 else ('*' if r.pvalues['ect']<0.1 else ''))
    print(f"  {h:>10}{r.params['ect']:>+10.4f}{r.tvalues['ect']:>8.2f}"
          f"{r.pvalues['ect']:>8.3f}{r.rsquared:>8.3f}  {sig}")
    horizons[h] = {'alpha': r.params['ect'], 'p': r.pvalues['ect'], 'r2': r.rsquared}

h12 = horizons[12]
if h12['p'] < 0.05:
    print(f"\n  ✓ 12-month α = {h12['alpha']:+.3f} (p={h12['p']:.3f}) — significant")
    print(f"    A 10% ECT predicts {abs(h12['alpha'])*10:.1f}% INR correction over 12 months")

# ── TIER 3: REGIME-CONDITIONAL ECM ──────────────────────────────
print(f"\n{'='*65}")
print("TIER 3 — REGIME-CONDITIONAL ECM")
print("="*65)

# 1-month with regime interaction
print("\n  A) 1-month, regime-split:")
d3 = ecm[['d_log_inr','ect_lag1','p_stress','d_log_dxy','d_log_brent']].dropna()
d3['ect_calm']   = d3['ect_lag1'] * (1 - d3['p_stress'])
d3['ect_stress'] = d3['ect_lag1'] * d3['p_stress']
X3 = sm.add_constant(d3[['ect_calm','ect_stress','d_log_dxy','d_log_brent']])
r3 = sm.OLS(d3['d_log_inr'], X3).fit(cov_type='HAC', cov_kwds={'maxlags':6})
print(f"  {'variable':<14}{'coef':>10}{'t-stat':>8}{'p-val':>8}")
for v in r3.params.index:
    sig = '***' if r3.pvalues[v]<0.01 else ('**' if r3.pvalues[v]<0.05 else ('*' if r3.pvalues[v]<0.1 else ''))
    print(f"  {v:<14}{r3.params[v]:>+10.5f}{r3.tvalues[v]:>8.2f}{r3.pvalues[v]:>8.3f} {sig}")
print(f"\n  α_calm   = {r3.params['ect_calm']:+.4f} — reversion in calm ({'✓ p<0.1' if r3.pvalues['ect_calm']<0.1 else '✗ insig'})")
print(f"  α_stress = {r3.params['ect_stress']:+.4f} — overshoot in stress (positive = divergence)")

# 6-month with regime interaction
print(f"\n  B) 6-month horizon, regime-split:")
fwd6 = ecm['log_inr'].shift(-6) - ecm['log_inr']
d6 = pd.DataFrame({'fwd6': fwd6, 'ect': ecm['ect'], 'p_stress': ecm['p_stress']}).dropna()
d6['ect_calm']   = d6['ect'] * (1 - d6['p_stress'])
d6['ect_stress'] = d6['ect'] * d6['p_stress']
X6 = sm.add_constant(d6[['ect_calm','ect_stress']])
r6 = sm.OLS(d6['fwd6'], X6).fit(cov_type='HAC', cov_kwds={'maxlags':6})
print(f"  {'variable':<14}{'coef':>10}{'t-stat':>8}{'p-val':>8}")
for v in r6.params.index:
    sig = '***' if r6.pvalues[v]<0.01 else ('**' if r6.pvalues[v]<0.05 else ('*' if r6.pvalues[v]<0.1 else ''))
    print(f"  {v:<14}{r6.params[v]:>+10.5f}{r6.tvalues[v]:>8.2f}{r6.pvalues[v]:>8.3f} {sig}")
print(f"  R² = {r6.rsquared:.3f}")

ac = r6.params['ect_calm']
if r6.pvalues['ect_calm'] < 0.05 and ac < 0:
    pct_close = abs(ac) * 100
    half_life = 6 * np.log(2) / abs(ac)
    print(f"\n  ✓ CALM-REGIME SIGNAL CONFIRMED:")
    print(f"    In calm periods, {pct_close:.0f}% of misalignment closes in 6 months")
    print(f"    Implied half-life ≈ {half_life:.0f} months")
    print(f"    Current ECT = {ecm['ect'].iloc[-1]:+.3f}, P(stress) = {ecm['p_stress'].iloc[-1]:.2f}")
    if ecm['p_stress'].iloc[-1] > 0.5:
        print(f"    → Currently in STRESS — signal is OFF (wait for calm)")
    else:
        expected = ac * ecm['ect'].iloc[-1] * 100
        print(f"    → Currently CALM — expected 6-month INR move: {expected:+.1f}%")

# ── Save enriched ECM dataset ───────────────────────────────────
ecm['fwd_1m']  = ecm['log_inr'].shift(-1) - ecm['log_inr']
ecm['fwd_6m']  = ecm['log_inr'].shift(-6) - ecm['log_inr']
ecm['fwd_12m'] = ecm['log_inr'].shift(-12) - ecm['log_inr']
ecm.to_csv(PROCESSED+'ecm_base.csv')

print(f"\n✓ Updated ecm_base.csv with forward returns")
print(f"  Ready for Cell 3: Out-of-sample signal test")

TIER 2 — BASIC 1-MONTH ECM
Sample: May2004 → Mar2026  (n=261)
  R² = 0.113
  variable            coef  t-stat   p-val
  const           +0.00263    2.13  0.0335 **
  ect_lag1        -0.00065   -0.03  0.9782 
  d_log_dxy       +0.20812    1.74  0.0826 *
  d_log_brent     -0.02276   -2.06  0.0396 **
  d_rrd           -0.00123   -0.82  0.4101 
  d_vix           +0.00017    0.49  0.6261 

  α = -0.0007 (p=0.978)
  ⚠ Insignificant at monthly frequency — consistent with Meese-Rogoff.
  Testing longer horizons and regime conditioning...

MULTI-HORIZON: Δlog(INR)_{{t:t+h}} = c + α_h · ECT_t
  h (months)         α  t-stat   p-val      R²
           1   +0.0174    0.57   0.566   0.002  
           3   -0.0170   -0.18   0.858   0.000  
           6   -0.0932   -0.61   0.542   0.007  
           9   -0.2977   -1.67   0.095   0.041  *
          12   -0.4501   -3.04   0.002   0.073  ***

  ✓ 12-month α = -0.450 (p=0.002) — significant
    A 10% ECT predicts 4.5% INR correction over 12 months

TIER 3

In [3]:
# ============================================================
# 06_unified_ecm.ipynb | Cell 3
# Out-of-Sample Signal Test (the credibility test)
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import statsmodels.api as sm

PROCESSED = '../data/processed/'
ecm = pd.read_csv(PROCESSED+'ecm_base.csv', index_col=0, parse_dates=True)
ecm.index = pd.to_datetime(ecm.index).to_period('M').to_timestamp()

# ── Train/test split ────────────────────────────────────────────
SPLIT = pd.Timestamp('2019-01-01')
train = ecm[ecm.index < SPLIT].dropna(subset=['ect','fwd_12m'])
test  = ecm[ecm.index >= SPLIT].dropna(subset=['ect','fwd_12m'])

print("="*65)
print("CELL 3 — OUT-OF-SAMPLE SIGNAL TEST")
print("="*65)
print(f"  In-sample  : {train.index[0].strftime('%b%Y')} → {train.index[-1].strftime('%b%Y')}  (n={len(train)})")
print(f"  Out-of-sample: {test.index[0].strftime('%b%Y')} → {test.index[-1].strftime('%b%Y')}  (n={len(test)})")

# ── In-sample estimation ────────────────────────────────────────
X_tr = sm.add_constant(train['ect']); y_tr = train['fwd_12m']
is_model = sm.OLS(y_tr, X_tr).fit(cov_type='HAC', cov_kwds={'maxlags':12})
alpha_is = is_model.params['ect']
print(f"\n  In-sample 12-month model:")
print(f"    α = {alpha_is:+.4f} (p={is_model.pvalues['ect']:.4f})")
print(f"    R² = {is_model.rsquared:.3f}")

# ── Out-of-sample predictions ──────────────────────────────────
X_oos = sm.add_constant(test['ect'])
test = test.copy()
test['predicted_12m'] = is_model.predict(X_oos)
test['actual_12m']    = test['fwd_12m']

# Direction: predicted < 0 → expect INR appreciation; > 0 → expect depreciation
test['pred_direction'] = np.where(test['predicted_12m'] < 0, 'appreciate', 'depreciate')
test['actual_direction'] = np.where(test['actual_12m'] < 0, 'appreciate', 'depreciate')
test['correct'] = test['pred_direction'] == test['actual_direction']

hit_rate = test['correct'].mean() * 100
n_correct = test['correct'].sum()
n_total   = len(test)

print(f"\n{'='*65}")
print(f"OUT-OF-SAMPLE RESULTS")
print(f"{'='*65}")
print(f"  Directional accuracy: {n_correct}/{n_total} = {hit_rate:.1f}%")
print(f"  (random baseline: 50%)")

# Correlation between predicted and actual
corr = test['predicted_12m'].corr(test['actual_12m'])
print(f"  Rank correlation: {test['predicted_12m'].corr(test['actual_12m'], method='spearman'):.3f}")
print(f"  Pearson correlation: {corr:.3f}")

# ── Simple signal-based strategy ────────────────────────────────
# When model predicts appreciation (pred < 0), go "long INR" (= short USDINR)
# Return = -actual_12m (profit from INR strengthening)
# When model predicts depreciation, go "long USD"
# Return = +actual_12m
test['strategy_ret'] = np.where(test['predicted_12m'] < 0,
                                 -test['actual_12m'],   # long INR
                                 +test['actual_12m'])    # long USD

# Buy-and-hold USD (benchmark)
test['bh_ret'] = test['actual_12m']  # holding USD vs INR

cum_strat = test['strategy_ret'].cumsum()
cum_bh    = test['bh_ret'].cumsum()

print(f"\n  STRATEGY PERFORMANCE (cumulative log returns):")
print(f"    Signal-based: {cum_strat.iloc[-1]*100:+.1f}%")
print(f"    Buy-hold USD: {cum_bh.iloc[-1]*100:+.1f}%")
print(f"    Excess return: {(cum_strat.iloc[-1]-cum_bh.iloc[-1])*100:+.1f}%")

# Annualized information ratio
if test['strategy_ret'].std() > 0:
    ir = (test['strategy_ret'].mean() / test['strategy_ret'].std()) * np.sqrt(12)
    print(f"    Information ratio: {ir:.2f}")

# ── Rolling out-of-sample (expanding window) ────────────────────
print(f"\n{'='*65}")
print(f"EXPANDING WINDOW OUT-OF-SAMPLE")
print(f"{'='*65}")
min_train = 120  # minimum 10 years of training data
results = []
for i in range(min_train, len(ecm)):
    dt = ecm.index[i]
    if pd.isna(ecm.loc[dt, 'fwd_12m']): continue
    
    # Train on everything before this date
    tr = ecm.iloc[:i].dropna(subset=['ect','fwd_12m'])
    if len(tr) < min_train: continue
    
    X_t = sm.add_constant(tr['ect']); y_t = tr['fwd_12m']
    try:
        mod = sm.OLS(y_t, X_t).fit()
        pred = mod.params['const'] + mod.params['ect'] * ecm.loc[dt, 'ect']
        actual = ecm.loc[dt, 'fwd_12m']
        results.append({
            'date': dt, 'pred': pred, 'actual': actual,
            'correct': (pred < 0) == (actual < 0),
            'alpha_rolling': mod.params['ect'],
        })
    except: continue

oos = pd.DataFrame(results).set_index('date')
print(f"  Window: {oos.index[0].strftime('%b%Y')} → {oos.index[-1].strftime('%b%Y')}  (n={len(oos)})")
print(f"  Directional accuracy: {oos['correct'].mean()*100:.1f}%")
print(f"  Correlation (pred vs actual): {oos['pred'].corr(oos['actual']):.3f}")

# Rolling α stability
print(f"\n  Rolling α stability:")
print(f"    Mean: {oos['alpha_rolling'].mean():.3f}")
print(f"    Min:  {oos['alpha_rolling'].min():.3f}")
print(f"    Max:  {oos['alpha_rolling'].max():.3f}")
print(f"    → {'✓ Stable' if oos['alpha_rolling'].max() < 0 else '⚠ Flips sign in some windows'}")

# ── Year-by-year OOS breakdown ──────────────────────────────────
print(f"\n  Year-by-year OOS hit rate:")
print(f"  {'year':>6}{'n':>6}{'hit%':>8}{'α':>8}")
for yr in sorted(oos.index.year.unique()):
    sub = oos[oos.index.year == yr]
    print(f"  {yr:>6}{len(sub):>6}{sub['correct'].mean()*100:>8.0f}%{sub['alpha_rolling'].mean():>8.3f}")

# ── Current signal reading ──────────────────────────────────────
print(f"\n{'='*65}")
print(f"CURRENT SIGNAL")
print(f"{'='*65}")
latest = ecm.dropna(subset=['ect']).iloc[-1]
pred_now = is_model.params['const'] + is_model.params['ect'] * latest['ect']
print(f"  ECT = {latest['ect']:+.4f} ({abs(latest['ect'])*100:.1f}% {'weak' if latest['ect']>0 else 'strong'})")
print(f"  P(stress) = {latest['p_stress']:.2f}")
print(f"  Predicted 12-month Δlog(INR) = {pred_now*100:+.1f}%")
print(f"  → Model predicts INR {'APPRECIATION' if pred_now<0 else 'DEPRECIATION'} over next 12 months")
if latest['p_stress'] > 0.5:
    print(f"  ⚠ BUT currently in STRESS regime — reversion historically unreliable")
    print(f"    Conditional signal: WAIT for calm regime before acting")
else:
    print(f"  ✓ Currently in CALM regime — signal is actionable")

ecm.to_csv(PROCESSED+'ecm_base.csv')
print(f"\n✓ Saved: ecm_base.csv (final)")

CELL 3 — OUT-OF-SAMPLE SIGNAL TEST
  In-sample  : Apr2004 → Dec2018  (n=177)
  Out-of-sample: Jan2019 → Apr2025  (n=76)

  In-sample 12-month model:
    α = -0.4484 (p=0.0032)
    R² = 0.070

OUT-OF-SAMPLE RESULTS
  Directional accuracy: 70/76 = 92.1%
  (random baseline: 50%)
  Rank correlation: 0.443
  Pearson correlation: 0.493

  STRATEGY PERFORMANCE (cumulative log returns):
    Signal-based: +280.8%
    Buy-hold USD: +280.8%
    Excess return: +0.0%
    Information ratio: 3.99

EXPANDING WINDOW OUT-OF-SAMPLE
  Window: Apr2014 → Apr2025  (n=133)
  Directional accuracy: 82.7%
  Correlation (pred vs actual): 0.275

  Rolling α stability:
    Mean: -0.446
    Min:  -0.479
    Max:  -0.401
    → ✓ Stable

  Year-by-year OOS hit rate:
    year     n    hit%       α
    2014     9     100%  -0.452
    2015    12     100%  -0.425
    2016    12       8%  -0.418
    2017    12      75%  -0.409
    2018    12      75%  -0.439
    2019    12     100%  -0.455
    2020    12      50%  -0.470
 

In [4]:
# ============================================================
# 06_unified_ecm.ipynb | Cell 4
# Summary Dashboard + Final Model Reading
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np

PROCESSED = '../data/processed/'

ecm = pd.read_csv(PROCESSED+'ecm_base.csv', index_col=0, parse_dates=True)
ecm.index = pd.to_datetime(ecm.index).to_period('M').to_timestamp()
p2 = pd.read_csv(PROCESSED+'phase2_equilibrium_v2.csv', index_col=0, parse_dates=True)
p2.index = pd.to_datetime(p2.index).to_period('M').to_timestamp()
p3 = pd.read_csv(PROCESSED+'phase3_feer_v2.csv', index_col=0, parse_dates=True)
p3.index = pd.to_datetime(p3.index).to_period('M').to_timestamp()
p4 = pd.read_csv(PROCESSED+'phase4_beer_v2.csv', index_col=0, parse_dates=True)
p4.index = pd.to_datetime(p4.index).to_period('M').to_timestamp()

# ── Gather latest readings from each phase ──────────────────────
r2 = p2.iloc[-1]; dt2 = p2.index[-1]
r3 = p3.iloc[-1]; dt3 = p3.index[-1]
r4 = p4.iloc[-1]; dt4 = p4.index[-1]
re = ecm.dropna(subset=['ect']).iloc[-1]; dte = ecm.dropna(subset=['ect']).index[-1]

# Fix P(stress): use Phase 4's last available value
p_stress = p4['p_stress'].dropna().iloc[-1]
p_stress_dt = p4['p_stress'].dropna().index[-1]
stress_label = 'STRESS' if p_stress > 0.5 else 'CALM'

print("╔" + "═"*66 + "╗")
print("║" + " INR/USD HYBRID FAIR VALUE MODEL — CURRENT READING".center(66) + "║")
print("╚" + "═"*66 + "╝")
print(f"\n  Spot INR/USD: {re['inr_usd']:.2f} ({dte.strftime('%b %Y')})\n")

print("─"*68)
print("  FOUR-MODEL CONVERGENCE")
print("─"*68)
print(f"\n  {'Model':<28}{'Date':>10}{'Misalign':>10}{'Fair INR':>10}")
print(f"  {'─'*58}")
print(f"  {'Phase 2: REER structural':<28}{dt2.strftime('%b%Y'):>10}"
      f"{r2['misalignment_pct']:>+9.1f}%{r2['inr_fair_reer']:>10.2f}")
print(f"  {'Phase 3: FEER static':<28}{dt3.strftime('%b%Y'):>10}"
      f"{r3['feer_static_pct']:>+9.1f}%{r3['inr_fair_static']:>10.2f}")
print(f"  {'Phase 3: FEER conditional':<28}{dt3.strftime('%b%Y'):>10}"
      f"{r3['feer_cond_pct']:>+9.1f}%{r3['inr_fair_cond']:>10.2f}")
print(f"  {'Phase 4: BEER short-run':<28}{dt4.strftime('%b%Y'):>10}"
      f"{r4['beer_misalign']:>+9.1f}%{r4['inr_fair_beer']:>10.2f}")
print(f"  {'Phase 6: Composite ECM':<28}{dte.strftime('%b%Y'):>10}"
      f"{re['ect']*100:>+9.1f}%{re['composite_fair']:>10.2f}")

print(f"\n  Financing premium: {r3['financing_premium']:+.1f}pp")
print(f"  (= gap between structural sustainability and financing reality)")

print(f"\n─"*68)
print("  REGIME STATUS")
print("─"*68)
print(f"\n  Markov P(stress) = {p_stress:.2f} at {p_stress_dt.strftime('%b %Y')}  →  [{stress_label}]")
print(f"  Oil×DXY regime: Brent=${ecm['composite_fair'].index[-1]}", end="")
# Get latest brent and dxy from master
ml = pd.read_csv(PROCESSED+'master_v2_lagged.csv', index_col=0, parse_dates=True)
ml.index = pd.to_datetime(ml.index).to_period('M').to_timestamp()
latest_brent = ml['brent'].dropna().iloc[-1]
latest_dxy   = ml['dxy'].dropna().iloc[-1]
print(f"\r  Oil×DXY: Brent=${latest_brent:.0f} (norm=$72)  DXY={latest_dxy:.0f}")

print(f"\n─"*68)
print("  ECM SIGNAL")
print("─"*68)
print(f"\n  ECT = {re['ect']:+.4f} ({abs(re['ect'])*100:.1f}% {'undervalued' if re['ect']>0 else 'overvalued'})")
print(f"  12-month α = -0.450 (p=0.002, stable across all windows)")
# Recalculate prediction using stored α
alpha_12 = -0.450
const_12 = 0.038   # approximate from in-sample
pred_12  = const_12 + alpha_12 * re['ect']
print(f"  Predicted 12-month move: {pred_12*100:+.1f}%")
print(f"  Decomposition:")
print(f"    Trend component (const): {const_12*100:+.1f}% (structural depreciation)")
print(f"    Reversion component (α×ECT): {(alpha_12*re['ect'])*100:+.1f}% (equilibrium pull)")
if abs(pred_12) < 0.02:
    print(f"  → Forces roughly balance — model predicts INR ≈ STABLE")
elif pred_12 < 0:
    print(f"  → Reversion dominates — model predicts INR APPRECIATION")
else:
    print(f"  → Trend dominates — model predicts continued DEPRECIATION")

print(f"\n─"*68)
print("  CONDITIONAL ASSESSMENT")
print("─"*68)
if p_stress > 0.5:
    print(f"\n  ⚠  STRESS REGIME ACTIVE (P={p_stress:.2f})")
    print(f"     Historical evidence: reversion UNRELIABLE during stress.")
    print(f"     α_calm = -0.05 (marginal); α_stress = +0.06 (overshoot).")
    print(f"     RECOMMENDATION: Do not act on the reversion signal.")
    print(f"     Wait for P(stress) < 0.5 (regime transition to calm).")
    print(f"     Monitor: Brent normalization, FPI inflow resumption,")
    print(f"     reserve stabilization as transition indicators.")
else:
    print(f"\n  ✓  CALM REGIME (P={p_stress:.2f})")
    print(f"     Reversion signal is historically reliable at 6-12mo horizon.")
    print(f"     6-month calm α = -0.30 (p=0.067); 12-month α = -0.45 (p=0.002)")

print(f"\n─"*68)
print("  MODEL CREDIBILITY METRICS")
print("─"*68)
print(f"\n  Equilibrium:")
print(f"    REER-FEER composite cointegrated (ADF p=0.0000)")
print(f"    BEER cointegrated (ADF p=0.013)")
print(f"  Prediction:")
print(f"    12-month α stable: -0.40 to -0.48 across all expanding windows")
print(f"    Never flips sign (strongest evidence against overfitting)")
print(f"    Magnitude correlation (OOS): Pearson=0.49, Spearman=0.44")
print(f"  Limitations:")
print(f"    Monthly predictability absent (Meese-Rogoff consistent)")
print(f"    Directional accuracy inflated by INR trend depreciation")
print(f"    Regime conditioning marginal (p=0.07 at 6mo, not 0.05)")
print(f"    FEER conditional norm based on Q3 FY26 — pre-war oil data")

print(f"\n{'═'*68}")
print(f"  Model output files:")
print(f"    master_v2.csv           — unlagged dataset (live tracker)")
print(f"    master_v2_lagged.csv    — point-in-time dataset (backtest)")
print(f"    phase2_equilibrium_v2   — REER structural anchor")
print(f"    phase3_feer_v2          — FEER + conditional norm")
print(f"    phase4_beer_v2          — BEER + regime classification")
print(f"    beer_coefs_v2.json      — BEER coefficients")
print(f"    ecm_base.csv            — ECM dataset + forward returns")
print(f"{'═'*68}")

╔══════════════════════════════════════════════════════════════════╗
║         INR/USD HYBRID FAIR VALUE MODEL — CURRENT READING        ║
╚══════════════════════════════════════════════════════════════════╝

  Spot INR/USD: 93.55 (Apr 2026)

────────────────────────────────────────────────────────────────────
  FOUR-MODEL CONVERGENCE
────────────────────────────────────────────────────────────────────

  Model                             Date  Misalign  Fair INR
  ──────────────────────────────────────────────────────────
  Phase 2: REER structural       Apr2026     +8.7%     86.06
  Phase 3: FEER static           Oct2025     +3.3%     86.23
  Phase 3: FEER conditional      Oct2025     -2.4%     91.35
  Phase 4: BEER short-run        Mar2026    +14.7%     80.86
  Phase 6: Composite ECM         Apr2026     +8.2%     86.15

  Financing premium: +5.8pp
  (= gap between structural sustainability and financing reality)

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─